# GlucoTwin: 03. Model Training, Ensemble Evaluation & In-Silico Twin
**Team:** Aethan, Amity University Mumbai  
**Challenge:** Digital Twin Challenge 2026 — Happiest Health Summit  

This notebook evaluates:
1. PyTorch Attention-LSTM + XGBoost ensemble performance on 57,480 held-out test sequences.
2. Clinical confusion matrix, ROC-AUC, PR-AUC, and feature importance rankings.
3. Patient-specific Digital Twin state synchronization and 'What-If' intervention simulation.



In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.ensemble import GlucoTwinEnsemble
from src.digital_twin.twin_engine import DigitalTwin



## 1. Load Trained GlucoTwin Ensemble


In [ ]:
models_dir = PROJECT_ROOT / "data" / "processed" / "models"
ensemble = GlucoTwinEnsemble.load(models_dir)
print("Ensemble successfully loaded!")
print("  LSTM weights:", ensemble.lstm_weight)
print("  XGBoost weights:", ensemble.xgb_weight)



## 2. Benchmark Metrics Leaderboard


In [ ]:
metrics_path = models_dir / "evaluation_metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

# Display tabular comparison
comparison = pd.DataFrame({
    'Metric': ['ROC-AUC', 'PR-AUC (Avg Prec)', 'Accuracy', 'Sensitivity (Recall)', 'Specificity', 'F1-Score', 'Brier Score'],
    'LSTM (0.6)': [
        metrics['lstm_model']['roc_auc'],
        metrics['lstm_model']['pr_auc'],
        f"{metrics['lstm_model']['accuracy']*100:.2f}%",
        f"{metrics['lstm_model']['recall_sensitivity']*100:.2f}%",
        f"{metrics['lstm_model']['specificity']*100:.2f}%",
        metrics['lstm_model']['f1_score'],
        metrics['lstm_model']['brier_score']
    ],
    'XGBoost (0.4)': [
        metrics['xgboost_model']['roc_auc'],
        metrics['xgboost_model']['pr_auc'],
        f"{metrics['xgboost_model']['accuracy']*100:.2f}%",
        f"{metrics['xgboost_model']['recall_sensitivity']*100:.2f}%",
        f"{metrics['xgboost_model']['specificity']*100:.2f}%",
        metrics['xgboost_model']['f1_score'],
        metrics['xgboost_model']['brier_score']
    ],
    'GlucoTwin Ensemble': [
        metrics['glucotwin_ensemble']['roc_auc'],
        metrics['glucotwin_ensemble']['pr_auc'],
        f"{metrics['glucotwin_ensemble']['accuracy']*100:.2f}%",
        f"{metrics['glucotwin_ensemble']['recall_sensitivity']*100:.2f}%",
        f"{metrics['glucotwin_ensemble']['specificity']*100:.2f}%",
        metrics['glucotwin_ensemble']['f1_score'],
        metrics['glucotwin_ensemble']['brier_score']
    ]
})

comparison



## 3. Top Feature Importance (XGBoost Branch)


In [ ]:
fi_path = models_dir / "feature_importance.json"
with open(fi_path, "r", encoding="utf-8") as f:
    fi_data = json.load(f)

fi_df = pd.DataFrame(fi_data).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='importance', y='feature', palette='Blues_r')
plt.title("Top 15 Predictive Features for 2-Hour Glucose Spike", fontsize=13, fontweight='bold')
plt.xlabel("Importance Weight", fontsize=11)
plt.ylabel("Engineered Feature", fontsize=11)
plt.tight_layout()
plt.show()



## 4. In-Silico Digital Twin 'What-If' Simulation


In [ ]:
# Instantiate Digital Twin for Patient P001
patient_profile = {
    "patient_id": "P001",
    "name": "Rajesh Sharma",
    "age": 52,
    "gender": "M",
    "city": "Mumbai",
    "bmi": 28.4,
    "hba1c": 8.7,
    "fasting_glucose": 158.0,
    "diabetes_duration_years": 6.5,
    "medication": "Metformin + Sitagliptin",
    "comorbidities": "Hypertension, Dyslipidemia"
}

twin = DigitalTwin(patient_id="P001", ehr_data=patient_profile, model_ensemble=ensemble)

# Ingest recent streaming telemetry showing upward glucose surge
readings = [
    {"timestamp": "2026-10-15T12:00:00Z", "glucose": 155.0, "heart_rate": 74, "steps": 50, "activity_type": "Sedentary"},
    {"timestamp": "2026-10-15T12:05:00Z", "glucose": 168.0, "heart_rate": 78, "steps": 60, "activity_type": "Sedentary"},
    {"timestamp": "2026-10-15T12:10:00Z", "glucose": 182.0, "heart_rate": 80, "steps": 40, "activity_type": "Sedentary"}
]
twin.update_state(readings)

# Run counterfactual trial: What if patient takes a 20-min brisk walk?
sim_walk = twin.simulate_intervention(intervention_type="walk", intensity="moderate")

print("Expected Clinical Improvement:")
print(sim_walk['expected_improvement'])
print(f"Peak Reduction: {sim_walk['peak_reduction_mg_dl']} mg/dL")
print(f"Spike Averted: {sim_walk['spike_averted']}")



In [ ]:
# Plot Counterfactual In-Silico Trajectory
base_vals = [p['glucose'] for p in sim_walk['baseline_trajectory']]
walk_vals = [p['glucose'] for p in sim_walk['intervention_trajectory']]
time_steps = [f"+{i*10}m" for i in range(1, len(base_vals) + 1)]

plt.figure(figsize=(12, 5))
plt.plot(time_steps, base_vals, 'r--', lw=2.2, label='Baseline (No Intervention - Spike to >215 mg/dL)')
plt.plot(time_steps, walk_vals, 'g-', lw=2.5, label='Counterfactual: 20-min Brisk Walk (GLUT4 Disposal)')
plt.axhspan(70, 180, color='green', alpha=0.15, label='Euglycemic Safe Zone (70-180 mg/dL)')
plt.axhline(180, color='red', linestyle=':', alpha=0.8, label='Spike Threshold (180 mg/dL)')

plt.title("In-Silico Digital Twin 'What-If' Simulation — Patient P001", fontsize=13, fontweight='bold')
plt.xlabel("Forecast Horizon (Minutes)", fontsize=11)
plt.ylabel("Glucose (mg/dL)", fontsize=11)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

